# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 6: LLM Agents with LangChain</font>

# <font color="#003660">Model Context Protocol</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how to use the model context protocol (MCP) <br>
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [LangChain Academy](https://academy.langchain.com/)
* [Introduction to LangChain Agents](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
* [LangChain Docs (Python)](https://python.langchain.com/)

In [1]:
!pip install -U langchain langchain-community langchain-openai langchain-mcp-adapters "fastapi[standard]" fastmcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.8/376.8 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [5]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [6]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [18]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


Now we will have to download both models for this session. Run the code below.

In [7]:
!ollama pull qwen3:8b # takes around a minute

For this notebook we will need the main.py file. This file starts a FastAPI server that simultaneously is also a fastmcp mcp server.

In [8]:
import urllib
urllib.request.urlretrieve("https://raw.githubusercontent.com/olivermueller/amlta-2025/refs/heads/main/Session_06/main.py", "main.py")

('main.py', <http.client.HTTPMessage at 0x7bdcfcbd3350>)

In [23]:
import subprocess

process = subprocess.Popen(
    ["python", "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

In [25]:
!curl -X POST "http://localhost:8000/add_numbers?a=5&b=3"

{"result":8}

## What is Model Context Protocol (MCP)?

MCP (Model Context Protocol) is an open-source standard for connecting AI applications to external systems ([Anthropic PBC, 2025](https://modelcontextprotocol.io/docs/getting-started/intro)). It was initially introduced by researchers from Anthropic but is now an open-source standard used in all libraries.

It simply integrates different tools in one MCP server that runs with a specific protocol to provide all tools in the same way to LLM agents as illustrated below by [Hou et al., 2025](https://doi.org/10.48550/arXiv.2503.23278).

![image.png](https://github.com/olivermueller/amlta-2025/blob/main/Session_06/imgs/mcp.png?raw=true)


The mcp server is provided in Session_06/[main.py](https://github.com/olivermueller/amlta-2025/blob/25473e34f5d39ec9080416d08e015648bef58102/Session_06/main.py) and we have already downloaded it into our data in this Colab environment. Let's check it out.

## Running an MCP Client

In [12]:
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

This defines where our client searches for MCP servers. If you have multiple you can add them all together here.

In [26]:
client = MultiServerMCPClient(
    {
        "mathserver": {
            "transport": "streamable_http",  # HTTP-based remote server
            # Ensure you start your math server on port 8000
            "url": "http://localhost:8000/mcp",
        }
    }
)
tools = await client.get_tools()

In [27]:
system_prompt = """You are an advanced AI assistant that can perform various tasks using the provided tools.
Use the tools as needed to fulfill user requests accurately and efficiently."""

config = {
    "model": "qwen3:8b",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)

In [28]:
agent = create_agent(
    system_prompt=system_prompt,
    model=model,
    tools=tools,
)

In [29]:
math_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (231924892493213 + 1203190329398129381)? Use the tool"}]}
)
for m in math_response["messages"]:
    print(m.pretty_repr())

================================ Human Message =================================

what's (231924892493213 + 1203190329398129381)? Use the tool
================================== Ai Message ==================================
Tool Calls:
  add_numbers (call_xojg3zyj)
 Call ID: call_xojg3zyj
  Args:
    a: 231924892493213
    b: 1203190329398129400
================================= Tool Message =================================
Name: add_numbers

{"result":1203422254290622613}
================================== Ai Message ==================================

The sum of 231924892493213 and 1203190329398129381 is **1203422254290622613**.


/tmp/ipython-input-2176641863.py:1: RuntimeWarning: coroutine 'Pregel.ainvoke' was never awaited
  math_response = await agent.ainvoke(


So overall, everything stays the same but there are some asynchronous calls.